In [0]:
%pip install hotel_reservation-0.0.1-py3-none-any.whl

Processing ./hotel_reservation-0.0.1-py3-none-any.whl
INFO: pip is looking at multiple versions of mlflow-skinny[databricks] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of mlflow-skinny[databricks] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of databricks-agents to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of google-api-core to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 681.8/681.8 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 59.0 MB/s eta 0:00:00
  

In [0]:
%restart_python

In [0]:
import os
import time

import mlflow
import pandas as pd
import requests
from databricks import feature_engineering
from databricks.sdk import WorkspaceClient
from pyspark.sql import SparkSession

from hotel_reservation.config import ProjectConfig
from hotel_reservation.serving.feature_serving import FeatureServing

In [0]:
spark = SparkSession.builder.getOrCreate()

fe = feature_engineering.FeatureEngineeringClient()

In [0]:
w = WorkspaceClient()
os.environ["DBR_HOST"] = w.config.host
os.environ["DBR_TOKEN"] = w.tokens.create(lifetime_seconds=1200).token_value

In [0]:
# Load project config
config = ProjectConfig.from_yaml(config_path="../project_config.yml")
catalog_name = config.catalog_name
schema_name = config.schema_name
feature_table_name = f"{catalog_name}.{schema_name}.hotel_reservation_preds"
feature_spec_name = f"{catalog_name}.{schema_name}.return_predictions"
endpoint_name = "hotel-reservation-feature-serving"

In [0]:
train_set = spark.table(f"{catalog_name}.{schema_name}.train_set").toPandas()
test_set = spark.table(f"{catalog_name}.{schema_name}.test_set").toPandas()
df = pd.concat([train_set, test_set])

model = mlflow.sklearn.load_model(f"models:/{catalog_name}.{schema_name}.hotel_reservation_model_basic@latest-model")


preds_df = df[["Booking_ID", "no_of_weekend_nights", "no_of_week_nights"]]
preds_df["Predicted_BookingStatus"] = model.predict(df[config.cat_features + config.num_features])
preds_df = spark.createDataFrame(preds_df)

fe.create_table(
    name=feature_table_name, primary_keys=["Booking_ID"], df=preds_df, description="Hotel reservation booking status predictions feature table"
)

spark.sql(f"""
          ALTER TABLE {feature_table_name}
          SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
        """)

# Initialize feature store manager
feature_serving = FeatureServing(
    feature_table_name=feature_table_name, feature_spec_name=feature_spec_name, endpoint_name=endpoint_name
)


/local_disk0/.ephemeral_nfs/envs/pythonEnv-a5b5bc96-7882-47cb-ad04-39066c99cba3/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/spark-a5b5bc96-7882-47cb-ad04-39/.ipykernel/4534/command-8985512492252896-3185895276:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  preds_df["Predicted_BookingStatus"] = model.predict(df[config.cat_features + config.num_features])


In [0]:
# Create online table
# feature_serving.create_online_table()
online_store_name = "hotel-reservation-predictions"
feature_serving.create_or_update_online_table(online_store_name=online_store_name)

In [0]:
# Create feature spec
feature_serving.create_feature_spec()

In [0]:
# Deploy feature serving endpoint
feature_serving.deploy_or_update_serving_endpoint()

In [0]:
start_time = time.time()
serving_endpoint = f"{os.environ['DBR_HOST']}/serving-endpoints/{endpoint_name}/invocations"
response = requests.post(
    f"{serving_endpoint}",
    headers={"Authorization": f"Bearer {os.environ['DBR_TOKEN']}"},
    json={"dataframe_records": [{"Booking_ID": "INN33900"}]},
)

end_time = time.time()
execution_time = end_time - start_time

print("Response status:", response.status_code)
print("Reponse text:", response.text)
print("Execution time:", execution_time, "seconds")


Response status: 200
Reponse text: {"outputs": [{"no_of_weekend_nights": 1, "no_of_week_nights": 4, "Booking_ID": "INN33900"}]}
Execution time: 0.14042925834655762 seconds


In [0]:
# another way to call the endpoint

response = requests.post(
    f"{serving_endpoint}",
    headers={"Authorization": f"Bearer {os.environ['DBR_TOKEN']}"},
    json={"dataframe_split": {"columns": ["Booking_ID"], "data": [["INN25637"]]}},
)

print("Response status:", response.status_code)
print("Reponse text:", response.text)

Response status: 200
Reponse text: {"outputs": [{"no_of_weekend_nights": 1, "no_of_week_nights": 2, "Booking_ID": "INN25637"}]}
